In [8]:
# import os
# import time
# import random
# from collections import deque, defaultdict
# from datetime import datetime, timedelta, timezone
# from typing import Any, Dict, List, Optional, Set, Tuple

# import requests
# import pandas as pd

# # -----------------------
# # Config
# # -----------------------
# API_KEY = "RGAPI-dda02be5-4e95-43fd-bc7d-910e92895087"
# if not API_KEY:
#     raise RuntimeError("Missing RIOT_API_KEY. Set env: RIOT_API_KEY=RGAPI-...")

# SEED_GAME_NAME = "Mooncake"
# SEED_TAG_LINE  = "0208"      # 改成你的真实 tag

# REGION_ROUTING = "americas"  # NA: americas
# QUEUE_ID = 420               # Ranked Solo/Duo
# DAYS_BACK = 30               # 只抓近30天的“上一把”

# MAX_PLAYERS = 20_000         # unique puuid 上限（2w）
# MAX_QUEUE = 60_000           # BFS 队列上限（防止极端爆炸）
# MAX_MATCHES_PER_PLAYER = 1   # 每个玩家只抓上一把（固定=1）

# # Edge构造：同队全连接（有向）会比较多，但每局只有10人，仍可控
# BUILD_FULL_TEAM_EDGES = True
# SAMPLED_EDGES_PER_SRC = 2    # 如果你把 BUILD_FULL_TEAM_EDGES=False，就用采样

# # duo/premade edge 过滤（若 Riot 返回 partyId）
# FILTER_SAME_PARTY_EDGES = True

# # rate limit
# BASE_SLEEP = 0.12
# JITTER = 0.08

# OUT_DIR = "data_bfs_lastmatch_20k"
# # -----------------------

# HEADERS = {"X-Riot-Token": API_KEY}

# def sleep_jitter(mult: float = 1.0) -> None:
#     time.sleep(max(0.0, BASE_SLEEP * mult + random.random() * JITTER))

# class RiotHTTPError(RuntimeError):
#     pass

# def riot_get(url: str, params: Optional[dict] = None, max_retries: int = 8) -> Any:
#     for attempt in range(max_retries):
#         r = requests.get(url, headers=HEADERS, params=params, timeout=30)
#         if r.status_code == 200:
#             sleep_jitter()
#             return r.json()
#         if r.status_code == 429:
#             ra = r.headers.get("Retry-After")
#             wait = float(ra) if ra else (1.5 + attempt * 0.8)
#             time.sleep(wait)
#             continue
#         if 500 <= r.status_code < 600:
#             time.sleep(1.0 + attempt * 0.6)
#             continue
#         raise RiotHTTPError(f"HTTP {r.status_code} for {url}: {r.text[:240]}")
#     raise RiotHTTPError(f"Max retries exceeded for {url}")

# def unix_days_ago(days: int) -> int:
#     return int((datetime.now(timezone.utc) - timedelta(days=days)).timestamp())

# # -----------------------
# # Riot endpoints
# # -----------------------
# def account_by_riot_id(game_name: str, tag_line: str) -> dict:
#     url = f"https://{REGION_ROUTING}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}"
#     return riot_get(url)

# def last_match_id_by_puuid(puuid: str, start_time_unix: int, queue: int) -> Optional[str]:
#     url = f"https://{REGION_ROUTING}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
#     params = {"queue": queue, "startTime": start_time_unix, "start": 0, "count": 1}
#     ids = riot_get(url, params=params)
#     if not ids:
#         return None
#     return ids[0]

# def match_detail(match_id: str) -> dict:
#     url = f"https://{REGION_ROUTING}.api.riotgames.com/lol/match/v5/matches/{match_id}"
#     return riot_get(url)

# # -----------------------
# # Parsing & edge building
# # -----------------------
# def parse_match(md: dict) -> Tuple[str, int, int, List[dict]]:
#     info = md.get("info", {})
#     meta = md.get("metadata", {})
#     match_id = meta.get("matchId")
#     queue_id = int(info.get("queueId", -1))
#     game_start = int(info.get("gameStartTimestamp", 0))
#     participants_out = []

#     for p in info.get("participants", []):
#         participants_out.append({
#             "puuid": p.get("puuid"),
#             "teamId": int(p.get("teamId")),
#             "win": bool(p.get("win")),
#             # partyId 不一定存在，存在则可用来剔除 premade 边
#             "partyId": p.get("partyId", None),
#         })
#     return match_id, queue_id, game_start, participants_out

# def build_edges_and_players(match_id: str, queue_id: int, game_start: int, participants: List[dict]) -> Tuple[List[dict], List[dict]]:
#     team_map: Dict[int, List[dict]] = defaultdict(list)
#     for p in participants:
#         if p["puuid"]:
#             team_map[p["teamId"]].append(p)

#     edges: List[dict] = []
#     player_rows: List[dict] = []

#     for team_id, plist in team_map.items():
#         puuids = [x["puuid"] for x in plist]
#         win_map = {x["puuid"]: x["win"] for x in plist}
#         party_map = {x["puuid"]: x.get("partyId") for x in plist}

#         for pu in puuids:
#             player_rows.append({
#                 "puuid": pu,
#                 "matchId": match_id,
#                 "queueId": queue_id,
#                 "gameStartTimestamp": game_start,
#                 "teamId": team_id,
#                 "win": win_map[pu],
#             })

#         for src in puuids:
#             teammates = [t for t in puuids if t != src]
#             if not teammates:
#                 continue

#             if BUILD_FULL_TEAM_EDGES:
#                 chosen = teammates
#             else:
#                 random.shuffle(teammates)
#                 chosen = teammates[:SAMPLED_EDGES_PER_SRC]

#             for dst in chosen:
#                 # 可选：过滤同 party（duo/premade）边
#                 if FILTER_SAME_PARTY_EDGES:
#                     a = party_map.get(src)
#                     b = party_map.get(dst)
#                     if a is not None and b is not None and a == b:
#                         continue

#                 edges.append({
#                     "src_puuid": src,
#                     "dst_puuid": dst,
#                     "matchId": match_id,
#                     "queueId": queue_id,
#                     "gameStartTimestamp": game_start,
#                     "teamId": team_id,
#                     "src_win": win_map[src],
#                 })

#     return edges, player_rows

# def teammates_in_same_team(participants: List[dict], focal_puuid: str) -> List[str]:
#     focal_team = None
#     for p in participants:
#         if p["puuid"] == focal_puuid:
#             focal_team = p["teamId"]
#             break
#     if focal_team is None:
#         return []

#     mates = []
#     for p in participants:
#         if p["teamId"] == focal_team and p["puuid"] != focal_puuid and p["puuid"]:
#             mates.append(p["puuid"])
#     return mates

# # -----------------------
# # BFS crawl
# # -----------------------
# def main():
#     os.makedirs(OUT_DIR, exist_ok=True)
#     start_time_unix = unix_days_ago(DAYS_BACK)

#     acct = account_by_riot_id(SEED_GAME_NAME, SEED_TAG_LINE)
#     seed_puuid = acct["puuid"]
#     print(f"[seed] {SEED_GAME_NAME}#{SEED_TAG_LINE} puuid={seed_puuid}")

#     visited: Set[str] = set([seed_puuid])
#     crawled_match_for: Set[str] = set()  # 每个玩家只抓一次“上一把”
#     q = deque([seed_puuid])

#     all_edges: List[dict] = []
#     all_players: List[dict] = []

#     while q and len(visited) < MAX_PLAYERS:
#         puuid = q.popleft()
#         if puuid in crawled_match_for:
#             continue

#         # 1) 拿该玩家上一把 matchId
#         try:
#             mid = last_match_id_by_puuid(puuid, start_time_unix, QUEUE_ID)
#         except RiotHTTPError as ex:
#             print(f"[warn] matchlist failed for {puuid}: {ex}")
#             crawled_match_for.add(puuid)
#             continue

#         if not mid:
#             crawled_match_for.add(puuid)
#             continue

#         # 2) 抓 match detail
#         try:
#             md = match_detail(mid)
#         except RiotHTTPError as ex:
#             print(f"[warn] match detail failed {mid}: {ex}")
#             crawled_match_for.add(puuid)
#             continue

#         match_id, queue_id, game_start, parts = parse_match(md)
#         if queue_id != QUEUE_ID:
#             crawled_match_for.add(puuid)
#             continue

#         # 3) 写入 edges + player rows（胜率统计）
#         edges, players = build_edges_and_players(match_id, queue_id, game_start, parts)
#         all_edges.extend(edges)
#         all_players.extend(players)

#         # 4) 将“该玩家这一把的队友”加入队列（继续扩展）
#         mates = teammates_in_same_team(parts, puuid)
#         random.shuffle(mates)

#         for m in mates:
#             if len(visited) >= MAX_PLAYERS:
#                 break
#             if m not in visited:
#                 visited.add(m)
#                 if len(q) < MAX_QUEUE:
#                     q.append(m)

#         crawled_match_for.add(puuid)

#         # progress
#         if len(crawled_match_for) % 200 == 0:
#             print(f"[progress] crawled_players={len(crawled_match_for)} "
#                   f"unique_players={len(visited)} queue={len(q)} edges={len(all_edges)}")

#     print(f"[done-crawl] crawled_players={len(crawled_match_for)} unique_players={len(visited)} edges={len(all_edges)}")

#     # 5) 统计胜率（仅基于抓到的“上一把”集合，而不是一个月全量）
#     dfp = pd.DataFrame(all_players)
#     dfp = dfp[dfp["queueId"] == QUEUE_ID].copy()
#     stats = dfp.groupby("puuid")["win"].agg(games="count", wins="sum").reset_index()
#     stats["winrate"] = stats["wins"] / stats["games"]

#     # 6) edges df
#     dfe = pd.DataFrame(all_edges)
#     dfe = dfe[dfe["queueId"] == QUEUE_ID].copy()
#     if "gameStartTimestamp" in dfe.columns:
#         dfe["gameStartUTC"] = pd.to_datetime(dfe["gameStartTimestamp"], unit="ms", utc=True)

#     # 7) save
#     edges_path = os.path.join(OUT_DIR, "edges.csv")
#     players_path = os.path.join(OUT_DIR, "players.csv")
#     dfe.to_csv(edges_path, index=False)
#     stats.to_csv(players_path, index=False)

#     print(f"[saved] edges={len(dfe)} -> {edges_path}")
#     print(f"[saved] players={len(stats)} -> {players_path}")
#     print(f"[note] winrate是基于“每人上一把”的样本，不是整月全量；若要整月全量胜率，需要每人抓多场。")

# if __name__ == "__main__":
#     main()


In [3]:
import os
import requests

API_KEY = "RGAPI-7812f795-b57a-43a6-bdce-28c8646dd12e"
if not API_KEY:
    raise RuntimeError("Please set RIOT_API_KEY env var, e.g. export RIOT_API_KEY=RGAPI-...")

REGION_ROUTING = "americas"  # NA / BR / LAN / LAS usually use americas
HEADERS = {"X-Riot-Token": API_KEY}

def puuid_to_riot_id(puuid: str) -> str:
    """
    Convert puuid -> "GameName#TagLine" using Riot Account-V1.
    """
    url = f"https://{REGION_ROUTING}.api.riotgames.com/riot/account/v1/accounts/by-puuid/{puuid}"
    r = requests.get(url, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    data = r.json()
    game_name = data.get("gameName", "")
    tag_line = data.get("tagLine", "")
    if not game_name or not tag_line:
        # some accounts may not expose name/tag (rare), still return what we have
        return f"{game_name}#{tag_line}".strip("#")
    return f"{game_name}#{tag_line}"

if __name__ == "__main__":
    puuid = "gceotjHF0RIdXc9lx839oGPmr-jUV-b43gtpBoHFXq-dqRME-c8eEEkMEHeM7_ymSL1ddVaHtm8huw"
    print(puuid_to_riot_id(puuid))


bugenedkpines#pine


如果每把match是一个node,edge表示这场match中有个人和另一场match重复，带有胜负weight

In [ ]:
FIYG1XAoQqjHEdKLJRwuJU4ciPbFEJS7vp5ZWR4ayqAPl5Wxn7m1FXWfCgkgQuieUL4V4SmgnBQHTw	OwjAiJVOWfJ3jIJIO2Ocf4Snp-qPheNTl51TW5rojdbnrEi6urQ0Brr2VaPXLOULFjMwB3NSoxYVGw	无向的	6791			1.0	14	8	1770275139552	1770545393032	0.5714286